<a href="https://colab.research.google.com/github/naokityokoyama/fake_news_hdc/blob/main/FAKENEWS_HDC_ML_ngram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install torch-hd binhd unidecode num2words -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.0/361.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 18.6 MB/s eta 0:00:00


In [4]:
import zipfile
from unidecode import unidecode
import string
from num2words import num2words
import re
import numpy as np
import pandas as pd
from typing import Union, Literal
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torchhd
from torchhd import embeddings
from torchhd.classifiers import AdaptHD, OnlineHD

from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from binhd.embeddings import ScatterCode
from binhd.datasets import BaseDataset
from binhd.classifiers import BinHD, NeuralHD
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import time

import torch.nn.functional as F
import math

import warnings
warnings.filterwarnings("ignore")

In [7]:
#build dataset

def build_dataset(isot, covid, fever):
  if isot:
    df = pd.read_csv('/content/drive/MyDrive/uff/isot.csv')
    return df
  elif covid:
    df = pd.read_csv('/content/drive/MyDrive/uff/covid.csv')
    return df
  elif fever:
    df = pd.read_csv('/content/drive/MyDrive/uff/fever.csv')
    return df

In [12]:
df = build_dataset(isot=False, covid=False, fever=True)

In [13]:
df.shape

(109809, 3)

In [ ]:
def batch(dataset, batch=True, size=1000):
  # Definir o tamanho da amostra
  if batch:
    sample_size = size

    # Criar uma amostra balanceada
    dataset = dataset.groupby("target", group_keys=False).apply(lambda x: resample(x, n_samples=sample_size // dataset["target"].nunique(), random_state=42))
    dataset = dataset.reset_index(drop=True)
    return dataset

  else:
    return dataset

In [ ]:
df['target'].value_counts()

,count
target,
1,22851
0,21416


In [ ]:
df = batch(df, batch=True, size=1000)

In [ ]:
df.shape

(1000, 3)

Clean

In [ ]:
def n2w(texto:str)->str:
  padrao = r"\d+"
  numeros = re.findall(padrao, texto)
  for numero in numeros:
    extenso = num2words(numero, lang='pt')
    texto = texto.replace(numero, extenso)
  return texto

In [ ]:
for repet in tqdm(range(2)):  #bug para rodar 2x
  df['frase'] = df['frase'].str.lower()
  df['frase'] = df['frase'].str.replace(f"[{string.punctuation}]", "", regex=True)
  df['frase'] = df['frase'].apply(lambda x: ' '.join(x.split()))
  df['frase'] = df['frase'].str.replace('"', '').str.replace('\\', '')
  #df['frase'] = df['frase'].apply(n2w)
  df['frase'] = df['frase'].apply(unidecode)
  df['frase'] = df['frase'].fillna("").str.replace(r"\d+", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

  0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (device)
print("Using {} device".format(device))

MAX_INPUT_SIZE = 128
PADDING_IDX = 0

ASCII_A = ord("a")
ASCII_Z = ord("z")
ASCII_SPACE = ord(" ")
NUM_TOKENS = ASCII_Z - ASCII_A + 3  # a through z plus space and padding
print (ASCII_A, '--', ASCII_Z, '--', ASCII_SPACE, '--', NUM_TOKENS)

def char2int(char: str) -> int:
    """Map a character to its integer identifier"""
    ascii_index = ord(char)

    if ascii_index == ASCII_SPACE:
        # Remap the space character to come after "z"
        return ASCII_Z - ASCII_A + 1

    return ascii_index - ASCII_A


def transform(x: str) -> torch.Tensor:
    char_ids = x[:MAX_INPUT_SIZE]
    char_ids = [char2int(char) + 1 for char in char_ids.lower()]

    if len(char_ids) < MAX_INPUT_SIZE:
        char_ids += [PADDING_IDX] * (MAX_INPUT_SIZE - len(char_ids))

    return torch.tensor(char_ids, dtype=torch.long)

cuda
Using cuda device
97 -- 122 -- 32 -- 28


In [ ]:
#create X and y
lst = []
print ('size dataset',df.shape[0] )
for i in range(df.shape[0]):
  lst.append (np.array(transform(df['frase'][i])))

X = np.array(lst)
y = list(df['target'])

# Recalculate min_val and max_val after fixing transform
min_val = float(X.min())
max_val = float(X.max())
print('val_min :', min_val, 'val_max :', max_val)

size dataset 5975
val_min : 0.0 val_max : 27.0


In [ ]:
X.shape

(5975, 128)

In [ ]:
NUM_LEVELS = 100 #usado no record encoding
BATCH_SIZE = 100
num_features = 128
DIMENSIONS = 1024
CLASSES = df['target'].nunique()

min_val = float(X.min())
max_val = float(X.max())

SIZE = X.shape[1]
print('val_min :', min_val, 'val_max :', max_val)

model_BindHD = BinHD(DIMENSIONS, CLASSES)



val_min : 0.0 val_max : 27.0


In [ ]:
class NgramEncoder(nn.Module):
    """
    N-gram encoder compatível com a interface da classe Sinusoid para uso na NeuralHD.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        n: int = 4,
        vocab_size: int = 256,
        vsa: str = "MAP",
        requires_grad: bool = False,
        device: torch.device = None,
        dtype: torch.dtype = None
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super(NgramEncoder, self).__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.n = n
        self.vocab_size = vocab_size
        self.vsa = vsa

        # Para compatibilidade com a regeneração da NeuralHD
        self.weight = nn.parameter.Parameter(
            torch.empty((out_features, in_features), **factory_kwargs),
            requires_grad=requires_grad,
        )

        self.bias = nn.parameter.Parameter(
            torch.empty((1, out_features), **factory_kwargs),
            requires_grad=requires_grad,
        )

        # Encoder de símbolos para n-gramas
        self.symbol = embeddings.Random(
            vocab_size,
            out_features,
            padding_idx=PADDING_IDX,
            device=device,
            dtype=dtype
        )

        self.reset_parameters()

    def reset_parameters(self) -> None:
        """Inicializa os parâmetros para compatibilidade com regeneração."""
        nn.init.normal_(self.weight, 0, 1)
        self.weight.data.copy_(F.normalize(self.weight.data))
        nn.init.uniform_(self.bias, 0, 2 * math.pi)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass que primeiro transforma dados contínuos e depois aplica n-gramas.
        """
        # Aplicar transformação linear similar à Sinusoid
        projected = F.linear(x, self.weight)

        # Quantizar para índices discretos
        quantized = torch.clamp(
            ((projected + self.bias) * self.vocab_size / (2 * math.pi)).long(),
            0, self.vocab_size - 1
        )

        # Obter embeddings dos símbolos
        symbols = self.symbol(quantized)

        # Aplicar n-gramas
        sample_hv = torchhd.ngrams(symbols, n=self.n)

        # Normalizar
        result = torchhd.normalize(sample_hv)

        # Retornar como tensor VSA
        vsa_tensor = torchhd.functional.get_vsa_tensor_class(self.vsa)
        return result.as_subclass(vsa_tensor)

encode = NgramEncoder(num_features, DIMENSIONS)
encode = encode.to(device)

In [ ]:
#somente para o binhd
class NgramEncoder(nn.Module):
    def __init__(self, out_features, size):
        super(NgramEncoder, self).__init__()
        self.symbol = embeddings.Random(size, out_features, padding_idx=PADDING_IDX)
        #print(self.symbol)
    def forward(self, x):
        #print (x, x.shape)
        symbols = self.symbol(x)
        sample_hv = torchhd.ngrams(symbols, n=4)  #colocar no construtor
        return torchhd.normalize(sample_hv)

encode = NgramEncoder(DIMENSIONS, NUM_TOKENS)
encode = encode.to(device)

In [ ]:
num_samples = df.shape[0]
batch_size = 32

for i in tqdm(range(0, num_samples, batch_size)):
  X_ = X[i:i+batch_size]
  y_ = y[i:i+batch_size]

  with torch.no_grad():
      samples = torch.tensor(X_).to(device)
      labels = torch.tensor(y_).squeeze().to(device)

      X_hd = encode(samples)
  X_train, X_test, y_train, y_test = train_test_split(X_hd, labels, test_size=0.30, random_state = 42)
  # X_train_, X_test_, y_train_, y_test_ = train_test_split(X_hd, labels, test_size=0.30, random_state = 42)

  model_BindHD = BinHD(DIMENSIONS, CLASSES)
  with torch.no_grad():
      model_BindHD.fit(X_train,y_train)
      predictions = model_BindHD.predict(X_test)
      acc = accuracy_score(predictions, y_test)
      #print("BinHD: Accuracy = ", acc)

  0%|          | 0/187 [00:00<?, ?it/s]

RuntimeError: index_add_(): self (Char) and source (Float) must have the same scalar type

In [ ]:
# model_NeuralHD = NeuralHD(n_dimensions=128, n_classes=2, n_features=1000)
# with torch.no_grad():
#     model_NeuralHD.fit(X_train_,y_train_)
#     predictions = model_NeuralHD.predict(X_test_)
#     acc = accuracy_score(predictions, y_test_)
#     print("NeuralHD: Accuracy = ", acc)

In [ ]:
# Ativar
# class RecordEncoder(nn.Module):
#     def __init__(self, out_features, size, levels, low, high):
#         super(RecordEncoder, self).__init__()
#         self.position = embeddings.Random(size, out_features, vsa="BSC", dtype=torch.uint8)
#         self.value = ScatterCode(levels, out_features, low = low, high = high)

#     def forward(self, x):
#         sample_hv = torchhd.bind(self.position.weight, self.value(x))
#         sample_hv = torchhd.multiset(sample_hv)
#         return sample_hv

# record_encode = RecordEncoder(DIMENSION, SIZE, NUM_LEVELS, min_val, max_val)
# record_encode = record_encode.to(device)

# with torch.no_grad():
#     samples = torch.tensor(X).to(device)
#     labels = torch.tensor(y).squeeze().to(device)

#     X_hv = record_encode(samples)

# X_train, X_test, y_train, y_test = train_test_split(X_hv, labels, test_size=0.3, random_state = 0)

# model = BinHD(DIMENSION, CLASSES)

# with torch.no_grad():
    # model.fit(X_train,y_train)
    # predictions = model.predict(X_test)
    # acc = accuracy_score(predictions, y_test)
    # print("BinHD: Accuracy = ", acc)

Batch

In [ ]:
class train_hdc:
  def __init__(self, X: Union[float, int],
               y: Union[float, int],
               random:int = 42,
               test_size:float = 0.30,
               dimension:int=1000,
               batch_size:int=8,
               classes:int=2,
               features:int=128):
    self.X = X
    self.y = y
    self.random = random
    self.test_size = test_size
    self.dimension = dimension
    self.batch_size = batch_size
    self.classes = classes
    self.features = features

  def prepare_data(self)->str:
    X = self.X
    y = self.y
    test_size = self.test_size

    #X_train_bind, X_test_bind, y_train_bind, y_test_bind = train_test_split(X, y, test_size=test_size, random_state =42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state =42)

    train_dataset = BaseDataset(X_train, y_train)
    test_dataset = BaseDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=self.batch_size)
    test_loader = DataLoader(test_dataset, batch_size=self.batch_size)
    return train_loader, test_loader

  def train(self)-> str:
    #models
    DIMENSION = self.dimension
    CLASSES = self.classes
    BATCH_SIZE = self.batch_size
    FEATURES = self.features
    print ('BATCH SIZE ->', BATCH_SIZE)
    model_BindHD = BinHD(DIMENSION, CLASSES)
    model = model_BindHD.to(device) # Move model to the specified device

    #train BINHD
    data_train = self.prepare_data()[0]

    with torch.no_grad():
      lst_start = []
      lst_end = []


      for x_ , y_ in tqdm(data_train) :
        #start time
        start = time.perf_counter()
        #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        samples = torch.tensor(x_).to(device)
        labels = torch.tensor(y_).squeeze().to(device)

        X_Bind = encode(samples)
        X_Bind = X_Bind.clip(0,1) #
        X_Bind = X_Bind.to(torch.int8)

        model.fit(X_Bind,labels)
        trained_model = model
        end = time.perf_counter()

        lst_start.append(start)
        lst_end.append(end)
        print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
      return trained_model


  def test(self, model:str)->str:
    data_test = self.prepare_data()[1]
    model_name = model.__class__.__name__

    model_BinHD = model
    print ('model BinHD')

    lst_acuracia = []
    lst_f1 = []
    lst_precision = []
    lst_recall = []

    with torch.no_grad():
      for x_ , y_ in data_test:

        samples = torch.tensor(x_).to(device)
        labels = torch.tensor(y_).squeeze().to(device)


        X_Bind = encode(samples)
        #X_Bind = X_Bind.clip(0,1)
        X_Bind = X_Bind.to(torch.int8)
        predictions = model_BinHD.predict(X_Bind)

        # Ensure y_test and y_pred are at least 1D arrays
        y_pred = np.atleast_1d(np.array(predictions))
        y_test = np.atleast_1d(np.array(labels))

          #metrics
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)


          #add lst
        lst_f1.append(f1)
        lst_acuracia.append(accuracy)
        lst_precision.append(precision)
        lst_recall.append(recall)

    print(f"{'BinHD'}: Accuracy = ", np.mean(lst_acuracia), "F1 -> ", np.mean(lst_f1),
          'Precision-> ', np.mean(lst_precision), 'recall ->', np.mean(lst_recall))

In [ ]:
X.shape

(5975, 128)

In [ ]:
hdc = train_hdc(X, y, batch_size=1000)

In [ ]:
#train BindHD
start = time.perf_counter()
modelo_bindhd = hdc.train()
end = time.perf_counter()
print(f"Elapsed: {end - start:.6f} s")

BATCH SIZE -> 1000


  0%|          | 0/5 [00:00<?, ?it/s]

Time total: 1.038553 s
Time total: 1.798354 s
Time total: 2.546289 s
Time total: 3.300829 s
Time total: 3.486250 s
Elapsed: 3.515473 s


In [ ]:
#test
start = time.perf_counter()
hdc.test(model=modelo_bindhd)
end = time.perf_counter()
print(f"Time total: {end - start:.6f} s")

model BinHD
BinHD: Accuracy =  0.2652559899117276 F1 ->  0.39826655299801084 Precision->  0.24919363808252698 recall -> 0.9920634920634921
Time total: 1.513330 s


ML

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

In [ ]:
batch_size = 3000
num_samples = df.shape[0]

In [ ]:
def train(model:str):
  #start time
  start = time.perf_counter()
  lst_acc = []
  lst_f1 = []
  lst_recall = []
  lst_precision = []
  lst_start = []
  lst_end = []

  for i in tqdm(range(0, num_samples, batch_size)):
    X_ = X[i:i+batch_size]
    y_ = y[i:i+batch_size]

    samples = torch.tensor(X_).to(device)
    labels = torch.tensor(y_).squeeze().to(device)

    X_encoder = encode(samples.float())

    # Use y_ for stratification instead of y
    X_train, X_test, y_train, y_test = train_test_split(X_encoder, y_, test_size=0.30, random_state = 42)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # # METRICS
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    #end
    end = time.perf_counter()
    # #LIST ADD
    lst_acc.append(acc)
    lst_f1.append(f1)
    lst_precision.append(precision)
    lst_recall.append(recall)
    lst_start.append(start)
    lst_end.append(end)

  print ('-- REPORT --')
  print (f'-- {model.__class__.__name__} --')
  print ('ACCURACY ->', np.mean(lst_acc))
  print ('F1 ->', np.mean(lst_f1))
  print ('Precision ->', np.mean(lst_precision))
  print ('Recall ->', np.mean(lst_recall))
  print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
  print ('\n')
  print ('------------------------------------------------')

In [ ]:
model_RL = LogisticRegression()
model_RF = RandomForestClassifier()
model_SVC = SVC()
model_TREE = DecisionTreeClassifier()
model_KNN = KNeighborsClassifier()

In [ ]:
lst_models = [model_RL, model_RF,model_SVC, model_TREE, model_KNN]

In [ ]:
for i in tqdm(lst_models):
  train(i)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

-- REPORT --
-- LogisticRegression --
ACCURACY -> 0.69765833022272
F1 -> 0.31772360447907066
Precision -> 0.2915395566402278
Recall -> 0.3495247215719657
Time total: 38.154703 s


------------------------------------------------


  0%|          | 0/2 [00:00<?, ?it/s]

-- REPORT --
-- RandomForestClassifier --
ACCURACY -> 0.7908062709966406
F1 -> 0.30405405405405406
Precision -> 0.7555292259083728
Recall -> 0.22144782577853445
Time total: 32.833486 s


------------------------------------------------


...

-- REPORT --
-- SVC --
ACCURACY -> 0.746899962672639
F1 -> 0.0
Precision -> 0.0
Recall -> 0.0
Time total: 37.276640 s


------------------------------------------------


  0%|          | 0/2 [00:00<?, ?it/s]

-- REPORT --
-- DecisionTreeClassifier --
ACCURACY -> 0.7116648002986189
F1 -> 0.2913592667113794
Precision -> 0.2954705882352941
Recall -> 0.2874547776122579
Time total: 32.081550 s


------------------------------------------------


  0%|          | 0/2 [00:00<?, ?it/s]

-- REPORT --
-- KNeighborsClassifier --
ACCURACY -> 0.7140350877192982
F1 -> 0.3301907719609583
Precision -> 0.27818118948824344
Recall -> 0.46233241115130874
Time total: 31.458707 s


------------------------------------------------


TorchHD

In [ ]:
DIMENSIONS = 1024  # number of hypervector dimensions
BATCH_SIZE = 1  # for GPUs with enough memory we can process multiple

num_features = 128
num_classes = 2

In [ ]:
from torchhd.embeddings import Random, Level, Projection, Sinusoid, Density

In [ ]:
# encsin = Sinusoid(num_features, DIMENSIONS)
# encsin = encsin.to(device)

# encopro = Projection(num_features, DIMENSIONS)
# encopro = encopro.to(device)

# enclat = Level(num_features, DIMENSIONS)
# enclat = enclat.to(device)



In [ ]:
# class NgramEncoder(nn.Module):
#     def __init__(self, out_features, size):
#         super(NgramEncoder, self).__init__()
#         self.symbol = embeddings.Random(size, out_features, padding_idx=PADDING_IDX)
#         #print(self.symbol)
#     def forward(self, x):
#         #print (x, x.shape)
#         symbols = self.symbol(x)
#         sample_hv = torchhd.ngrams(symbols, n=4)  #colocar no construtor
#         return torchhd.normalize(sample_hv)

# encode = NgramEncoder(num_features, DIMENSIONS)
# encode = encode.to(device)

In [ ]:
import torch
import torchhd
from torchhd.datasets.isolet import ISOLET
from binhd.datasets import BaseDataset
from binhd.classifiers import BinHD

In [ ]:
classifiers = [
    "Vanilla",
    "AdaptHD",
    "OnlineHD",
    "NeuralHD",
    "DistHD",
    "CompHD",
    "SparseHD",
    "QuantHD",
    "LeHDC"

]



DIMENSIONS = 1024  # number of hypervector dimensions
BATCH_SIZE = 1  # for GPUs with enough memory we can process multiple

num_features = 128
num_classes = 2

In [ ]:
params = {
    "Vanilla": {},
    "AdaptHD": {
        "epochs": 2,
    },
    "OnlineHD": {
        "epochs": 2,
    },
    "NeuralHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "DistHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "CompHD": {},
    "SparseHD": {
        "epochs": 2,
    },
    "QuantHD": {
        "epochs": 2,
    },
    "LeHDC": {
        "epochs": 2
    }
}

In [ ]:
batch_size = 1000
num_samples = df.shape[0]

In [ ]:
def model_torch_hd(model_name):

  lst_acuracia = []
  lst_f1 = []
  lst_precision = []
  lst_recall = []
  lst_start = []
  lst_end = []
  for i in tqdm(range(0, num_samples, batch_size)):
    #start time
    start = time.perf_counter()

    x_ = X[i:i+batch_size]
    y_ = y[i:i+batch_size]
    samples = torch.tensor(x_).to(device)
    labels = torch.tensor(y_).squeeze().to(device)

    X_ = samples
    #X_ = encode(samples)
    X_ = X_.float()
    X_train, X_test, y_train, y_test = train_test_split(X_, labels, test_size=0.30, random_state = 42)

    train_dataset = BaseDataset(X_train, y_train)
    test_dataset = BaseDataset(X_test, y_test)

    train_ld = torch.utils.data.DataLoader(train_dataset)#, batch_size=BATCH_SIZE)
    test_ld = torch.utils.data.DataLoader(test_dataset)#, batch_size=BATCH_SIZE)

    model_cls = getattr(torchhd.classifiers, model_name)
    model: torchhd.classifiers.Classifier = model_cls(
          num_features, DIMENSIONS, num_classes, device=device, **params[model_name]
      )

    # Convert X_test to float before predicting
    X_test_tmp = torch.tensor(X_test).to(device)
    X_test_tmp_float = X_test_tmp.float()

    #encoder
    model.encoder = encode

    model.fit(train_ld)


    accuracy = model.accuracy(test_ld)

    y_pred = model.predict(X_test_tmp_float)
    y_pred = y_pred.cpu().numpy()
    y_test = y_test.cpu().numpy()


    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    #end
    end = time.perf_counter()

    lst_acuracia.append(accuracy)
    lst_f1.append(f1)
    lst_precision.append(precision)
    lst_recall.append(recall)
    lst_start.append(start)
    lst_end.append(end)

  print ('model', model_cls.__name__)
  print(f"Testing accuracy of {(np.mean(lst_acuracia) * 100):.3f}%")
  print(f"Testing F1 of {(np.mean(lst_f1) * 100):.3f}%")
  print(f"Testing RECALL of {(np.mean(lst_recall) * 100):.3f}%")
  print(f"Testing PRECISION of {(np.mean(lst_precision) * 100):.3f}%")
  print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
  print ('\n')
  print ('------------------------------------------------')

In [ ]:
for i in classifiers:
  model_torch_hd(i)
  print (i)


  0%|          | 0/6 [00:00<?, ?it/s]

model Vanilla
Testing accuracy of 74.356%
Testing F1 of 34.702%
Testing RECALL of 51.006%
Testing PRECISION of 34.583%
Time total: 5.384825 s


------------------------------------------------
Vanilla


  0%|          | 0/6 [00:00<?, ?it/s]


fit: 100%|██████████| 2/2 [00:01<00:00,  1.11it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.19it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.09it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.17it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.14it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.21it/s]


model AdaptHD
Testing accuracy of 84.983%
Testing F1 of 14.394%
Testing RECALL of 16.667%
Testing PRECISION of 12.667%
Time total: 12.734834 s


------------------------------------------------
AdaptHD


...


fit: 100%|██████████| 1/1 [00:00<00:00,  1.74it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]


model QuantHD
Testing accuracy of 73.578%
Testing F1 of 34.877%
Testing RECALL of 52.183%
Testing PRECISION of 33.695%
Time total: 6.628960 s


------------------------------------------------
QuantHD


  0%|          | 0/6 [00:00<?, ?it/s]


fit: 100%|██████████| 2/2 [00:02<00:00,  1.20s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.03s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.03s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.04s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.06s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.03s/it]


model LeHDC
Testing accuracy of 81.427%
Testing F1 of 23.945%
Testing RECALL of 26.721%
Testing PRECISION of 22.766%
Time total: 18.893706 s


------------------------------------------------
LeHDC


In [ ]:
for i in tqdm(classifiers):
  model_torch_hd(i)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

model Vanilla
Testing accuracy of 90.372%
Testing F1 of 90.716%
Testing RECALL of 91.614%
Testing PRECISION of 89.906%
Time total: 92.607311 s


------------------------------------------------


  0%|          | 0/45 [00:00<?, ?it/s]



fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.22it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.19it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.05it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.08it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.20it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.19it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.24it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.27it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.27it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:01<00:01,  1.01s/it]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.10it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.31it/s]

fit: 100

...



fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.60it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.75it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.60it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]


fit:   0%|          | 0/1 [00:00<?, ?it/s]

fit: 100%|██████████| 1/1 [00:00<00:00,  1.50it/s]


fit:   0%|          | 0/1 [0

model QuantHD
Testing accuracy of 82.816%
Testing F1 of 82.272%
Testing RECALL of 78.248%
Testing PRECISION of 87.033%
Time total: 116.955000 s


------------------------------------------------


  0%|          | 0/45 [00:00<?, ?it/s]



fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:01<00:01,  1.08s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.11s/it]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.07it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.03it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:01<00:01,  1.02s/it]

fit: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.09it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.05it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.04it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.04it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.02it/s]

fit: 100%|██████████| 2/2 [00:01<00:00,  1.01it/s]


fit:   0%|          | 0/2 [00:00<?, ?it/s]

fit:  50%|█████     | 1/2 [00:00<00:00,  1.05it/s]

fit: 100

model LeHDC
Testing accuracy of 72.670%
Testing F1 of 72.099%
Testing RECALL of 75.203%
Testing PRECISION of 76.588%
Time total: 155.985397 s


------------------------------------------------
